# 📓 Day 7: Advanced Prompt Engineering

Welcome to Day 7 of the AI Engineer & Agentic AI Engineer Bootcamp. Today, we bridge the gap between AI and software systems by exploring **Advanced Prompt Engineering**.

### 🎯 Learning Objectives
1. Initialize models with strict **System Instructions** using the Gemini SDK.
2. Enforce **Structured Outputs (JSON)** using **Pydantic** schema constraints.
3. Set up **Function Calling (Tool Use)** to execute local Python code based on LLM decisions.
4. Build a **Resume Parser** that converts raw text into structured databases.

### 🛑 Prerequisites
- Gemini API Key loaded in your environment (`GEMINI_API_KEY`)
- Python packages: `google-generativeai`, `pydantic`, `python-dotenv`

--- 
## 🛠️ Environment Configuration
Let's load our environment variables and configure the Gemini client.

In [1]:
import os
from dotenv import load_dotenv
from google import genai

# Load environment variables from .env
load_dotenv()

# Read API Key
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "❌ GEMINI_API_KEY not found. Please check your .env file."
    )

# Initialize Gemini Client
client = genai.Client(api_key=api_key)

print("✅ Gemini API configured successfully.")

✅ Gemini API configured successfully.


--- 
## 💂️ 1. System Instructions

System Instructions allow developers to specify permanent rules, personas, and safety boundaries for the model. Let's see how system instructions control behavior.

In [5]:
from google.genai import types

system_instruction = """
You are Ramesh Kaka, the canteen owner at a college in Kolhapur.
You speak in a warm, friendly mix of Hindi and occasionally some English.
Always greet the student with 'Beta' and mention a local snack like Samosa or Chai.
Keep answers extremely brief and do not use professional corporate jargon.
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What is the menu for today?",
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.7,
    ),
)

print(response.text)

Beta, namaste! Aaj ka menu, Samosa ke saath... Poha, Misal Pav, and Vada Pav. Chai-coffee bhi hai!


In [4]:
from google.genai import types

system_instruction = """
You are James, a friendly food court associate at a busy shopping mall in England.
You speak in a warm, professional British English tone with a welcoming attitude.
Always greet customers with 'Hello there!' and casually mention a popular menu item like a Chicken Wrap, Fish & Chips, or Flat White coffee.
Keep your responses concise, polite, and customer-focused.
Avoid slang, technical explanations, or unnecessary detail.
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What is the menu for today?",
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.7,
    ),
)

print(response.text)

Hello there! Today we've got our delicious Chicken Wraps flying out, and of course, all our usual favourites. You can see the full menu just above the counter, or I can tell you about any specific dishes you fancy.


--- 
## 🧾 2. Structured Outputs & JSON Mode

To build software applications, our Python code needs predictable data structures. We use **Pydantic** to declare the schema, and configure the model to output *only* valid JSON conforming to that schema.

In [6]:
from typing import List
from pydantic import BaseModel, Field
from google.genai import types


# -----------------------------
# Pydantic Schema
# -----------------------------
class OrderDetails(BaseModel):
    student_name: str = Field(
        description="Full name of the student making the order."
    )

    items: List[str] = Field(
        description="List of food items ordered."
    )

    total_bill_inr: float = Field(
        description="Total bill in INR. Samosa = 15, Chai = 10, Maggi = 30."
    )

    is_credit_purchase: bool = Field(
        description="Whether the order is on credit."
    )

    credit_reason: str | None = Field(
        description="Reason for taking credit, if any."
    )


# -----------------------------
# User Input
# -----------------------------
user_input = """
I am Rohit from CSE branch.
I ordered 2 samosas and 1 cup of tea.
Please write this down in my credit diary.
I will pay Ramesh Kaka on Saturday after getting pocket money.
"""


# -----------------------------
# Generate Structured Output
# -----------------------------
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"""
Parse the following canteen order and return ONLY structured data.

{user_input}
""",
    config=types.GenerateContentConfig(
        temperature=0,
        response_mime_type="application/json",
        response_schema=OrderDetails,
    ),
)


# -----------------------------
# JSON Output
# -----------------------------
print("Raw JSON:")
print(response.text)


# -----------------------------
# Parsed Pydantic Object
# -----------------------------
print("\nParsed Object:")
print(response.parsed)


# -----------------------------
# Access Individual Fields
# -----------------------------
order = response.parsed

print("\nStudent:", order.student_name)
print("Items:", order.items)
print("Bill:", order.total_bill_inr)
print("Credit:", order.is_credit_purchase)
print("Reason:", order.credit_reason)

Raw JSON:
{"student_name":"Rohit","items":["Samosa","Samosa","Chai"],"total_bill_inr":40,"is_credit_purchase":true,"credit_reason":"Will pay Ramesh Kaka on Saturday after getting pocket money."}

Parsed Object:
student_name='Rohit' items=['Samosa', 'Samosa', 'Chai'] total_bill_inr=40.0 is_credit_purchase=True credit_reason='Will pay Ramesh Kaka on Saturday after getting pocket money.'

Student: Rohit
Items: ['Samosa', 'Samosa', 'Chai']
Bill: 40.0
Credit: True
Reason: Will pay Ramesh Kaka on Saturday after getting pocket money.


Let's parse the JSON text directly into a Python Pydantic Object. This makes it instantly ready to store in our databases.

In [ ]:
import json

# Convert text JSON into Pydantic validation object
try:
    data = json.loads(response.text)
    parsed_order = OrderDetails(**data)
    
    print(f"Parsed Python Object Name: {parsed_order.student_name}")
    print(f"Ordered Items: {parsed_order.items}")
    print(f"Is Udhaari? {parsed_order.is_credit_purchase}")
    print(f"Total Bill: ₹{parsed_order.total_bill_inr}")
except Exception as e:
    print(f"Parsing failed: {e}")

--- 
## ✏️ 3. Mini Exercise: Canteen Chat Order Parser

**Your Task:** Write a Pydantic schema called `CustomerTicket` to parse customer feedback into JSON. The schema must extract:
- `customer_name`: Name of the sender (use "Unknown" if missing)
- `mood`: Must be either `HAPPY`, `FRUSTRATED`, or `NEUTRAL`
- `order_rating`: Score from 1 to 5 (None if not mentioned)
- `feedback_summary`: A single sentence summarizing the message

Then run the model to evaluate the customer ticket: *"Hey, I am Amit. My delivery boy took 1 hour to reach my hostel, the food was cold and spilled. Very bad service, 1 star rating."*

In [ ]:
# Write your schema here!
class CustomerTicket(BaseModel):
    # Define fields here
    customer_name: str = Field(default="Unknown")
    mood: str = Field(description="Must be HAPPY, FRUSTRATED, or NEUTRAL")
    order_rating: Optional[int] = Field(default=None, description="Score from 1 to 5")
    feedback_summary: str = Field(description="A single sentence summarizing the feedback")

# Initialize model and generate response
ticket_text = "Hey, I am Amit. My delivery boy took 1 hour to reach my hostel, the food was cold and spilled. Very bad service, 1 star rating."

# Hint: If using the legacy google-generativeai SDK, use response_schema=clean_schema(CustomerTicket.model_json_schema())
# to prevent 'default' field schema validation errors.
# response = ...
# print(response.text)

--- 
## 📞 4. Function Calling (Tool Use)

Function calling teaches LLMs how to act as coordinators. They inspect the user prompt, identify if they have a python function to solve it, formulate the arguments, and request execution.

In [ ]:
from google.genai import types


# ----------------------------------------
# Local Python Tool
# ----------------------------------------
def query_udhaari_balance(student_name: str) -> str:
    """Queries the canteen ledger diary (khata book) for a student's outstanding balance."""

    database = {
        "rohit": "₹250 outstanding. Last purchased 3 samosas on credit.",
        "amit": "₹0 balance. All clear!",
        "riya": "₹75 outstanding. Last purchased 1 Maggi on credit."
    }

    student_name = student_name.lower().strip()

    if student_name in database:
        return database[student_name]

    return "No active ledger entry found."


# ----------------------------------------
# Generate Response with Tool Calling
# ----------------------------------------
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="How much money does amit owe Ramesh Kaka?",
    config=types.GenerateContentConfig(

        tools=[
            query_udhaari_balance
        ],

        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=False
        ),

        temperature=0
    )
)

print(response.text)

In [ ]:
from google import genai
from google.genai import types
import os

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

# ----------------------------------------
# Local Python Tool
# ----------------------------------------
def query_udhaari_balance(student_name: str) -> str:
    """
    Queries the canteen ledger diary (khata book)
    for a student's outstanding credit balance.

    Args:
        student_name: Name of the student.
    """

    database = {
        "rohit": "₹250 outstanding. Last purchased 3 samosas on credit.",
        "amit": "₹0 balance. All clear!",
        "riya": "₹75 outstanding. Last purchased 1 Maggi on credit."
    }

    student_name = student_name.lower().strip()

    if student_name in database:
        return database[student_name]

    return "No active ledger entry found."


# ----------------------------------------
# Generate Response
# ----------------------------------------
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="How much money does rohit owe Ramesh Kaka?",
    config=types.GenerateContentConfig(

        tools=[query_udhaari_balance],

        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=False
        ),

        temperature=0
    )
)

print(response.text)

In [ ]:
# Define a local mock function representing database lookup
def query_udhaari_balance(student_name: str) -> str:
    """Queries the canteen ledger diary (khata book) for a student's outstanding credit balance.
    
    Args:
        student_name: The name of the student to look up.
    """
    # Mock database lookup
    database = {
        "rohit": "₹250 outstanding. Last purchased 3 samosas on credit.",
        "amit": "₹0 balance. All clear!",
        "riya": "₹75 outstanding. Last purchased 1 Maggi on credit."
    }
    name_key = student_name.lower().strip()
    if name_key in database:
        return f"Balance for {student_name}: {database[name_key]}"
    return f"Student '{student_name}' has no active ledger entry in the canteen credit book."

# Register the function as a tool in GenerativeModel
tool_model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    tools=[query_udhaari_balance] # Provide function lists
)

# Start a chat session with automatic execution enabled
# The SDK automatically runs query_udhaari_balance and submits results back to Gemini
chat = tool_model.start_chat(enable_automatic_function_calling=True)

response = chat.send_message("How much money does Rohit owe Ramesh Kaka?")
print("Final Chat Agent Response:")
print(response.text)

--- 
## 🏭 5. Industry Challenge: Resume Parser CLI

**Scenario**: Your placement cell wants to automate the resume screening process for students. You are building a helper script that takes raw resume text, parses it into structured JSON, and checks if the student meets a criteria.

**Task**: Create a complete script that extracts applicant metadata using structured outputs.

In [ ]:
# Helper function to recursively clean the Pydantic schema for the legacy Gemini API
def clean_schema(schema: dict) -> dict:
    if isinstance(schema, dict):
        cleaned = {}
        for k, v in schema.items():
            if k not in ["title", "default", "additionalProperties", "definitions", "$defs"]:
                cleaned[k] = clean_schema(v)
        if "anyOf" in schema:
            subschemas = schema["anyOf"]
            non_null_type = next((s for s in subschemas if s.get("type") != "null"), None)
            if non_null_type:
                cleaned.update(clean_schema(non_null_type))
                cleaned["nullable"] = True
            cleaned.pop("anyOf", None)
        return cleaned
    elif isinstance(schema, list):
        return [clean_schema(item) for item in schema]
    return schema

class ProjectDetails(BaseModel):
    title: str = Field(description="Title of the project")
    technologies_used: List[str] = Field(description="Programming languages/frameworks used")
    description: str = Field(description="1-sentence summary of project")

class ResumeSchema(BaseModel):
    candidate_name: str = Field(description="Full name of applicant")
    email: str = Field(description="Candidate email address")
    skills: List[str] = Field(description="Key technical skills (e.g. Python, SQL, Git)")
    projects: List[ProjectDetails] = Field(description="List of projects listed in the resume")
    highest_qualification: str = Field(description="Highest degree (e.g., B.Tech CSE, BCA)")

resume_text = """
RAKESH SHARMA
Email: rakesh.sharma@email.com
Degree: B.Tech in Computer Science (2026)

TECHNICAL SKILLS:
Python, JavaScript, PostgreSQL, Git, UV Package Manager

PROJECTS:
1. Student Grading App - Built using Python and SQLite. Automated grading logs for 500 students.
2. Smart Bus Kiosk - Created a ReactJS web application for city bus schedules using public transport REST APIs.
"""

parser_model = genai.GenerativeModel("gemini-1.5-flash")

response = parser_model.generate_content(
    f"Parse this resume: {resume_text}",
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",
        response_schema=clean_schema(ResumeSchema.model_json_schema()),
        temperature=0.0
    )
)

print("Parsed Resume Schema Output:")
parsed_data = json.loads(response.text)
print(json.dumps(parsed_data, indent=2))

--- 
## 💡 6. Summary & Key Takeaways
- **System Instructions** set strict behavioral guidelines that remain constant across chats.
- **Structured Output** forces Gemini to output valid JSON compliant with a Pydantic schema. This eliminates regex parsing and makes LLM outputs software-friendly.
- **Function Calling** acts as the routing mechanism for tools, enabling AI agents to connect to local databases and third-party API gateways.
- Set `temperature=0.0` for all validation, extraction, and tool execution routines to ensure predictability.